# Naming the emotion in a short message

One English sentence goes in. The model picks one emotion: **sadness, joy, love, anger, fear, or surprise**. It also says how sure it is.

We start with where the messages come from. The model is already trained. We load it when we reach the network, so the early cells stay about the data.

## 1. Where the messages come from

These are English tweets. In 2018, Saravia and co-authors collected them for a paper called CARER.

They did not ask a person to label each tweet. A tweet that ended with an emotion hashtag, such as `#joy` or `#sadness`, kept that hashtag as its label. This is called distant supervision. The label is useful, but it is noisy: the writer chose the tag, and the tag is not always a careful judgment.

They then cleaned the tweet. The text was lowercased. User names and links were set aside. The hashtag used as the label was removed, so the model cannot cheat by reading the tag.

The paper used eight emotions. We use the later teaching copy, which keeps six: sadness, joy, love, anger, fear, and surprise. It is published on Hugging Face as `dair-ai/emotion`. We take the prepared split: 16,000 messages to train, 2,000 to check the model while training, and 2,000 kept aside for the final test. Each example is the message and a number from 0 to 5.

In [ ]:
from data import load_texts
from emotions import EMOTIONS

for split in ("train", "validation", "test"):
    messages, split_labels = load_texts(split)
    print(f"{split:12} {len(messages):5} messages")

train_messages, train_labels = load_texts("train")
print()
print("one stored message")
print(" text: ", train_messages[0])
print(" label:", train_labels[0], "->", EMOTIONS[train_labels[0]])

The tweets are already cleaned. From here we load the trained model once, then keep going.

In [ ]:
from collections import Counter
from pathlib import Path

import torch
from IPython.display import Image, display

from data import Vocabulary, load_texts, tokenize
from emotions import EMOTIONS
from model import EmotionLSTM

CHECKPOINT = Path("outputs/model.pt")
if not CHECKPOINT.is_file():
    raise FileNotFoundError("The trained model is missing.")
device = torch.device("cpu")
saved = torch.load(CHECKPOINT, map_location=device, weights_only=False)
vocabulary = Vocabulary(saved["vocab"])
network = EmotionLSTM(len(vocabulary), pad_id=vocabulary.pad_id).to(device)
network.load_state_dict(saved["state_dict"])
network.eval()
print(f"trained model loaded    validation accuracy {saved['val_accuracy']:.1%}")

## 2. How the classes are spread

Each example is a short text and a label from 0 to 5. Joy and sadness are common. Surprise is rare. A high accuracy can still miss a rare emotion, so later we look at every class.

In [ ]:
texts, labels = load_texts("train")
counts = Counter(labels)
shown = [
    "i feel so sad and hopeless",
    "im feeling happy and well",
    "i feel romantic too",
    "im feeling really quite angry",
    "im feeling afraid",
    "i feel surprised because i didnt expect it",
]
print(f"{len(texts)} training messages\n")
for class_id, name in enumerate(EMOTIONS):
    print(f"{class_id}  {name:10}  {counts[class_id]:5}   {shown[class_id]}")

## 3. A sentence becomes numbers

The network cannot read letters. We lowercase the text, keep the words, and swap each word for an id from the training vocabulary. An unknown word becomes `<unk>`. `<pad>` is only used to line up a batch.

In [ ]:
sentence = "i feel romantic too"
tokens = tokenize(sentence)
ids = vocabulary.encode(sentence)
print("words:     ", tokens)
print("ids:       ", ids)
print("vocabulary:", len(vocabulary), "words")

## 4. Three layers

1. **Embedding.** Each word id picks a vector of 128 numbers. Those vectors are learned on this dataset.
2. **LSTM.** It reads the vectors from left to right. A cell state carries memory. Three gates decide what to forget, what to write, and what to show.
3. **Linear layer.** The hidden state after the last real word becomes 6 scores, one per emotion.

The biggest score wins.

In [ ]:
token_ids = torch.tensor([ids])
lengths = torch.tensor([len(ids)])
with torch.no_grad():
    vectors = network.embedding(token_ids)
    logits = network(token_ids, lengths)
print(network)
print()
print("word ids ", tuple(token_ids.shape))
print("embedding", tuple(vectors.shape), "  batch, words, 128")
print("logits   ", tuple(logits.shape), "     batch, 6 emotions")
print("scores   ", [round(value, 2) for value in logits[0].tolist()])
print("names    ", EMOTIONS)

## 5. The test result

The test set has 2,000 messages. The model did not train on them, and we did not use them to choose which trained version to keep.

**Accuracy** is the share of messages we got right. **Macro F1** is the average across the six emotions, so surprise counts as much as joy.

In [ ]:
print(Path("outputs/classification_report.txt").read_text())
display(Image("outputs/confusion_matrix.png"))

The diagonal is the correct answers. Joy and love share a lot of words, so the model mixes them. Surprise has the fewest training messages and the weakest score.

## 6. Three new sentences

Same saved model. Top three emotions for each line.

In [ ]:
def classify(text):
    word_ids = vocabulary.encode(text)
    batch = torch.tensor([word_ids])
    length = torch.tensor([len(word_ids)])
    with torch.no_grad():
        probabilities = torch.softmax(network(batch, length), dim=1)[0]
    top = torch.topk(probabilities, k=3)
    print(text)
    for score, index in zip(top.values.tolist(), top.indices.tolist()):
        print(f"  {score:6.1%}  {EMOTIONS[index]}")
    print()

classify("I can't believe I got the job")
classify("I feel so alone tonight")
classify("This delay makes me furious")

## What to say at the end

We turn each message into word ids. An embedding and an LSTM read those words in order. A linear layer scores six emotions. On the test set, accuracy is 86.9%. The mistakes are mostly between emotions that use similar words.